# Built with DINOv3

This notebook uses weights derived from Meta's **DINOv3**. DINOv3 is distributed under the DINOv3 License, included in the attached dataset as `DINOV3_LICENSE.md`.

Other pretrained weights used here carry their own separate licences.

# Combined: our ensemble + the public pipeline (prize-safe)

The public side is a copy of `yamadan96/rsna-knee-d4-public0946` (code by Mattia Angeli, Jiwei Liu, dreaddevelopment / Roman Tamrazov, Pilkwang and others) with Stage 3 removed (RadImageNet is non-commercial). Its weights are CC0 datasets; DINOv2-small (Meta) is Apache-2.0. It runs first in its own kernel, then our ensemble, then a per-finding rank blend of the two.


In [ ]:
import time as _kt
_KERNEL_T0 = _kt.time()
import sys, glob, json, subprocess, traceback, importlib.util
from pathlib import Path
hits = sorted(glob.glob('/kaggle/input/**/shared_code.py', recursive=True))
if not hits:
    raise SystemExit('shared_code.py not found. tree: '
                     + str(sorted(glob.glob('/kaggle/input/**/*', recursive=True))[:200]))
W = Path(hits[0]).parent
sys.path.insert(0, str(W))
# Kaggle sometimes mounts the PREVIOUS dataset version right after an upload (cost a 43 min
# stress run on 1 Oct). Stop in seconds instead; the publisher sees this and pushes again.
import hashlib
_eng = hashlib.sha1((W / 'shared_code.py').read_bytes()).hexdigest()
if _eng != '45058f226e03e53cf7d46711b5035a60a9d58885':
    Path('engine_mismatch.txt').write_text(_eng)
    raise RuntimeError('ENGINE MISMATCH: mounted %s, expected 45058f226e03e53cf7d46711b5035a60a9d58885' % _eng)
print('engine', _eng)
subs = sorted(glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True))
if not subs:
    raise SystemExit('competition data not found under /kaggle/input')
COMP = Path(subs[0]).parent
MEMBERS = json.loads((W / 'members.json').read_text())
BLEND = None
_bw = W / 'blend_weights.json'
if _bw.exists():
    BLEND = json.loads(_bw.read_text())
    print('per-class blend weights found for', len(BLEND['members']), 'members')
else:
    print('no blend_weights.json -> flat average')
print('weights:', W, '| comp:', COMP)
print('members:', [m['name'] for m in MEMBERS])

In [ ]:
# the test set mixes JPEG Lossless and JPEG 2000 in with the raw series and pydicom
# reads neither on its own. no internet here, so the wheels ride along in the dataset.
wheels = sorted(glob.glob(str(W / '*.whl')))
try:
    import pylibjpeg, openjpeg, libjpeg
    print('dicom decoders already present')
except ImportError:
    if wheels:
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                            '--no-deps', '-q', *wheels], capture_output=True, text=True)
        print('installed decoders rc=', r.returncode, r.stderr[-300:])
    else:
        print('WARNING: no decoder wheels, compressed series will fail')

In [ ]:
import shared_code as sc, pandas as pd, torch
series = pd.read_csv(COMP / 'test_series.csv')
sub = pd.read_csv(COMP / 'sample_submission.csv')
TASKS = [c for c in sub.columns if c != 'StudyInstanceUID']
print(len(sub), 'studies to score')

def bail(why):
    # always leave a valid file behind: 0.5 everywhere scores about 0.5, while no file
    # at all scores nothing and still costs one of the five daily attempts
    traceback.print_exc()
    print('FALLING BACK TO 0.5:', why)
    out = sub.copy()
    out[TASKS] = 0.5
    out.to_csv('submission.csv', index=False)

bail('nothing has run yet')          # overwritten the moment real work lands

In [ ]:
# member 0 owns the loader geometry; the others only re-normalise the same batch
try:
    CFGS = [json.loads((W / ('m%d_config.json' % m['i'])).read_text()) for m in MEMBERS]
    idx, files_by_series = sc.scan_dicom_index(COMP / 'test_series', series)
    plan = sc.plan_slots(idx, CFGS[0])
    print(len(plan), 'studies planned of', len(sub))
except Exception:
    bail('scan/plan failed'); raise SystemExit(0)

In [ ]:
# each member's model_def under its OWN module name - every run's train.py defines a
# class called SliceModel, so one shared `model_def` would hand every member the same
# architecture and silently score the wrong thing
def _load_def(i):
    spec = importlib.util.spec_from_file_location('m%d_def' % i, W / ('m%d_def.py' % i))
    mod = importlib.util.module_from_spec(spec)
    sys.modules['m%d_def' % i] = mod
    spec.loader.exec_module(mod)
    return mod

# Load weights whichever side of torch.compile they were saved from. The notebook does
# not import shared_code, so this mirrors its helper: strip a leading '_orig_mod.' (left
# by a checkpoint written from a compiled model) and load into the unwrapped module.
def _load_model_state(model, sd):
    if any(k.startswith('_orig_mod.') for k in sd):
        sd = {(k[len('_orig_mod.'):] if k.startswith('_orig_mod.') else k): v
              for k, v in sd.items()}
    return getattr(model, '_orig_mod', model).load_state_dict(sd)

# SPECS say how to rebuild a member from nothing. The second T4 is driven by a separate
# PROCESS, because threads share one interpreter lock and measured 2x slower per card, and
# a built model holding CUDA tensors cannot be pickled to a child. The child rebuilds its
# own members from these same three files. Plain data only, so it survives the spawn.
SPECS = []
try:
    MODELS = []
    for m, cfg in zip(MEMBERS, CFGS):
        i = m['i']
        SPECS.append({'i': int(i), 'dir': str(W), 'cfg': cfg})
        model = _load_def(i).build_model(cfg)
        ck = torch.load(W / ('m%d_model.pt' % i), map_location='cpu', weights_only=False)
        _load_model_state(model, ck['model'])
        MODELS.append(model.eval())
        print('m%d %-30s epoch %s score %s' % (i, m['name'], ck.get('epoch'), ck.get('score')))
except Exception:
    bail('model load failed'); raise SystemExit(0)

In [ ]:
# STRESS repeats the public studies to imitate the hidden set's size, so the
# per-study rate is MARGINAL rather than dominated by one-time warm-up. It also
# suppresses submission.csv, because the repeated rows are not a real submission.
# 0 = a normal run.
STRESS = 128

todo = sub[['StudyInstanceUID']]
if STRESS:
    reps = -(-STRESS // len(todo))
    todo = pd.concat([todo] * reps, ignore_index=True).head(STRESS)
    print('STRESS MODE:', len(todo), 'studies')

# ONE decode pass shared by every member. Decoding is ~9s/study against <1s of GPU,
# so looping predict_dicom per member would be ~4x9s = over the 9h cap; this is
# ~9s + N x <1s. Averaging happens in logit space, as measured offline.
import time as _time
_t0 = _time.time()
try:
    # reorder the saved matrix into MEMBERS order, and into the task order this
    # notebook actually submits in - both are checked, because a silent mismatch here
    # blends the wrong model into the wrong finding and still writes a valid-looking file
    _W = None
    if BLEND:
        _names = [m['name'] for m in MEMBERS]
        assert _names == BLEND['members'], (
            'member order differs: %s vs %s' % (_names, BLEND['members']))
        _ti = [BLEND['tasks'].index(t) for t in TASKS]
        _W = [[row[j] for j in _ti] for row in BLEND['matrix']]
        print('using per-class blend weights')
    # _grouped, not the plain one: members that disagree on loader geometry (size,
    # mm_per_px, n_slices, slots, windowing) cannot share a decode pass, so it decodes once
    # per geometry GROUP and blends at the end. With every member on one geometry this does
    # exactly what predict_dicom_ensemble did, in one pass, so it is safe as the default.
    pred = sc.predict_dicom_ensemble_grouped(CFGS, MODELS, todo, plan, idx, files_by_series,
                                             batch_size=1, weights=_W, model_specs=SPECS)
    # the 9h cap is enforced on the HIDDEN test set, which is larger than what runs
    # here, so the number that has to be extrapolated is s/study, not the total
    _el = _time.time() - _t0
    _rate = _el / max(len(todo), 1)
    print('INFERENCE  %.1f s for %d studies  =  %.2f s/study  |  9h cap fits ~%d studies'
          % (_el, len(todo), _rate, int(9 * 3600 / max(_rate, 1e-9))))
    # timing.json is written by predict_dicom_ensemble and holds the decode/per-member
    # split. Echo it so it is in the log too, then leave the file as a kernel output for
    # runtime_budget.py to ingest.
    try:
        print('TIMING ' + open('timing.json').read().replace(chr(10), ' '))
    except Exception:
        pass
    if not STRESS:
        out = sub[['StudyInstanceUID']].merge(pred, on='StudyInstanceUID', how='left')
        out[TASKS] = out[TASKS].astype(float).fillna(0.5).clip(0, 1)
        assert len(out) == len(sub), '%d rows for %d studies' % (len(out), len(sub))
        out.to_csv('submission.csv', index=False)
        print(out[TASKS].describe().T[['mean', 'min', 'max']])
except Exception:
    bail('inference failed')
pd.read_csv('submission.csv').head()

In [ ]:
# PUBLIC PIPELINE FIRST (prize-safe copy, Stage 3 RadImageNet removed), in its own kernel. Whatever happens
# here, our ensemble still runs and submission.csv falls back to it alone.
import os, time, shutil, signal
# STRESS comes from the inference cell (ours ran FIRST)
PUBLIC_W = 0.5
PUBLIC_CAP_H = min(7.5, (8.2 * 3600 - (time.time() - _KERNEL_T0)) / 3600)
print('PUBLIC SIDE after ours: %.2f h used, cap %.2f h' % ((time.time() - _KERNEL_T0) / 3600, PUBLIC_CAP_H), flush=True)
if not STRESS and Path('submission.csv').exists():
    shutil.copy('submission.csv', 'submission_ours.csv')      # the public kernel deletes submission.csv
for _nm in ('MODELS', 'SPECS'):
    globals().pop(_nm, None)                                  # free our weights before the public kernel loads
import gc as _gc
_gc.collect()
try:
    import torch as _tch
    _tch.cuda.empty_cache()
except Exception:
    pass
PUBLIC_OVERLAP = False
WORK = Path('/kaggle/working')
PUB_CSV = WORK / 'public_side.csv'
PUB_OK = False
_pub = {'status': 'not run', 'stress': STRESS, 'weight': PUBLIC_W, 'cap_h': PUBLIC_CAP_H}
_pub_map = {}
_pub_t0 = time.time()
try:
    if PUBLIC_CAP_H < 0.5:
        raise RuntimeError('skipped: %.2f h left after our ensemble' % PUBLIC_CAP_H)
    PUB_CSV.unlink(missing_ok=True)
    import pandas as _pd
    _penv = dict(os.environ)
    _penv['RSNA_TIME_BUDGET_S'] = str(int(PUBLIC_CAP_H * 3600))
    _penv.update({'RSNA_PARALLEL_COAT_READERS': '1', 'RSNA_SKIP_COAT_READERS': 'global96,repairv1'})
    _t = _pd.read_csv(COMP / 'test.csv', dtype={'StudyInstanceUID': str})
    _pub_ids = _t.StudyInstanceUID.tolist()
    if STRESS:
        # their code refuses repeated study IDs, so the stress cohort gets fake IDs whose folders point at real ones
        _fake = Path('/tmp/pub_stress_root')
        shutil.rmtree(_fake, ignore_errors=True)
        (_fake / 'test_series').mkdir(parents=True)
        for _it in COMP.iterdir():
            if _it.name not in ('test.csv', 'test_series.csv', 'test_series', 'sample_submission.csv'):
                (_fake / _it.name).symlink_to(_it, target_is_directory=_it.is_dir())
        _ts = _pd.read_csv(COMP / 'test_series.csv', dtype={'StudyInstanceUID': str, 'SeriesInstanceUID': str})
        _ss = _pd.read_csv(COMP / 'sample_submission.csv', dtype={'StudyInstanceUID': str})
        _rt, _rs, _rss = [], [], []
        for _k in range(STRESS):
            _r = _pub_ids[_k % len(_pub_ids)]
            _f = '%s.%d' % (_r, 9000 + _k)
            _pub_map[_f] = _r
            for _src, _acc in ((_t, _rt), (_ts, _rs), (_ss, _rss)):
                _row = _src[_src.StudyInstanceUID == _r].copy()
                _row['StudyInstanceUID'] = _f
                _acc.append(_row)
            (_fake / 'test_series' / _f).symlink_to(COMP / 'test_series' / _r, target_is_directory=True)
        _pd.concat(_rt).to_csv(_fake / 'test.csv', index=False)
        _pd.concat(_rs).to_csv(_fake / 'test_series.csv', index=False)
        _pd.concat(_rss).to_csv(_fake / 'sample_submission.csv', index=False)
        _penv['RSNA_COMP_ROOT'] = str(_fake)
        _pub_ids = list(_pub_map)
        print('PUBLIC STRESS: %d fake studies over %d real ones' % (len(_pub_ids), len(set(_pub_map.values()))))
    _pnb = WORK / 'public_pipeline.ipynb'
    shutil.copy(W / 'public_pipeline.ipynb', _pnb)
    _cmd = [sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute', str(_pnb),
            '--output', 'public_pipeline_executed.ipynb', '--output-dir', str(WORK),
            '--ExecutePreprocessor.timeout=-1', '--ExecutePreprocessor.kernel_name=python3']
    print('PUBLIC SIDE start, cap %.2f h%s' % (PUBLIC_CAP_H, ', OVERLAPPED with our ensemble' if PUBLIC_OVERLAP else ''), flush=True)
    _pub_fh = open(WORK / 'public_log.txt', 'w')
    _proc = subprocess.Popen(_cmd, cwd=str(WORK), env=_penv, stdout=_pub_fh, stderr=subprocess.STDOUT,
                             start_new_session=True)
except Exception as _e:
    traceback.print_exc()
    _proc = None
    _pub['status'] = 'failed: %s: %s' % (type(_e).__name__, str(_e)[:500])


def _pub_finish():
    # wait for the public kernel (until its cap), validate public_side.csv, clean up, write the receipt
    global PUB_OK
    if _pub.get('finished'):
        return
    _pub['finished'] = True
    try:
        if _proc is None:
            raise RuntimeError(_pub['status'])
        try:
            _rc = _proc.wait(timeout=max(1.0, _pub_t0 + PUBLIC_CAP_H * 3600 + 600 - time.time()))
        except subprocess.TimeoutExpired:
            # the whole process group: nbconvert AND the kernel it started, or the GPUs stay held
            os.killpg(_proc.pid, signal.SIGKILL)
            _proc.wait()
            _rc = 'timeout'
        _pub_fh.close()
        _pub['returncode'] = _rc
        if _rc != 0:
            raise RuntimeError('public notebook ended with %s' % _rc)
        import pandas as _pd
        _p = _pd.read_csv(PUB_CSV, dtype={'StudyInstanceUID': str})
        _lab = [c for c in _p.columns if c != 'StudyInstanceUID']
        _want = [c for c in _pd.read_csv(COMP / 'sample_submission.csv', nrows=1).columns if c != 'StudyInstanceUID']
        assert _lab == _want, 'public columns %s' % _lab
        assert sorted(_p.StudyInstanceUID) == sorted(_pub_ids), 'public study set differs from test.csv'
        import numpy as _npp
        assert _npp.isfinite(_p[_lab].to_numpy(float)).all(), 'public side has non-finite values'
        _pub.update(status='ok', rows=len(_p))
        PUB_OK = True
    except Exception as _e:
        traceback.print_exc()
        if not str(_pub['status']).startswith('failed'):
            _pub['status'] = 'failed: %s: %s' % (type(_e).__name__, str(_e)[:500])
        try:
            _pub['log_tail'] = (WORK / 'public_log.txt').read_text(errors='replace')[-3000:]
        except Exception:
            pass
    _pub['seconds'] = round(time.time() - _pub_t0, 1)
    _pub['overlap'] = PUBLIC_OVERLAP
    _pub['order'] = 'last'
    for _d in ('/tmp/rsna_pixels_v559', '/kaggle/temp/rsna_pixels_v559', '/tmp/pub_stress_root',
               str(WORK / '_coat_env'), str(WORK / '_g96_env'), str(WORK / '_d4_env'), str(WORK / 'd4_input')):
        shutil.rmtree(_d, ignore_errors=True)
    (WORK / 'public_pipeline.ipynb').unlink(missing_ok=True)
    try:
        _pub['gpu_mem_after'] = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader'],
                                               capture_output=True, text=True).stdout.strip()
    except Exception:
        pass
    (WORK / 'public_receipt.json').write_text(json.dumps(_pub, indent=1, default=str))
    print('PUBLIC SIDE %s in %.1f min (%s rows) | GPU memory after: %s'
          % (_pub['status'][:200], _pub['seconds'] / 60, _pub.get('rows'), _pub.get('gpu_mem_after')), flush=True)


if not PUBLIC_OVERLAP:
    _pub_finish()
if not STRESS and Path('submission_ours.csv').exists():
    shutil.copy('submission_ours.csv', 'submission.csv')     # ours back in place before the blend


In [ ]:
# BLEND per finding in rank space: (1 - w) x rank(ours) + w x rank(public). Ours alone if the public side failed;
# the public side alone if ours fell back to 0.5.
import numpy as _np
if not Path('submission.csv').exists():          # the public kernel's start-up deletes it; never end without one
    bail('no submission.csv before the blend')
_pub_finish()                                    # overlap: wait for the public kernel here (no-op otherwise)
_bl = {'mode': 'ours only', 'weight': PUBLIC_W, 'public_status': _pub['status'], 'overlap': PUBLIC_OVERLAP}

def _rank_blend(ours, theirs, w):
    theirs = theirs.set_index('StudyInstanceUID').reindex(ours.StudyInstanceUID.astype(str))
    assert theirs[TASKS].notna().all().all(), 'public side misses some studies'
    out = ours[['StudyInstanceUID']].copy()
    corr = {}
    for t in TASKS:
        wt = float(w[t]) if isinstance(w, dict) else float(w)
        ro = ours[t].rank(method='average', pct=True).to_numpy()
        rt = theirs[t].rank(method='average', pct=True).to_numpy()
        out[t] = (1 - wt) * ro + wt * rt
        corr[t] = round(float(_np.corrcoef(ro, rt)[0, 1]), 4) if len(ro) > 2 else None
    assert _np.isfinite(out[TASKS].to_numpy(float)).all()
    return out, corr

try:
    if PUB_OK:
        _theirs = pd.read_csv(PUB_CSV, dtype={'StudyInstanceUID': str})
        if STRESS:
            # dry run on the real public studies (first fake copy of each): proves the path, writes no submission
            _theirs['StudyInstanceUID'] = _theirs.StudyInstanceUID.map(_pub_map)
            _theirs = _theirs.drop_duplicates('StudyInstanceUID')
            _ours = pred.drop_duplicates('StudyInstanceUID').reset_index(drop=True)
            _ours['StudyInstanceUID'] = _ours.StudyInstanceUID.astype(str)
            _prev, _bl['rank_corr'] = _rank_blend(_ours, _theirs, PUBLIC_W)
            _prev.to_csv('stress_blend_preview.csv', index=False)
            _bl.update(mode='stress dry run', rows=len(_prev))
        else:
            _ours = pd.read_csv('submission.csv', dtype={'StudyInstanceUID': str})
            _ours.to_csv('submission_ours.csv', index=False)
            _neutral = float((_ours[TASKS].to_numpy(float) == 0.5).mean())
            _w = 1.0 if _neutral > 0.5 else PUBLIC_W
            _out, _bl['rank_corr'] = _rank_blend(_ours, _theirs, _w)
            assert len(_out) == len(sub) and set(_out.StudyInstanceUID) == set(sub.StudyInstanceUID.astype(str))
            _out.to_csv('submission.csv', index=False)
            _bl.update(mode='public only (ours fell back)' if _w == 1.0 else 'blended', rows=len(_out),
                       ours_neutral_share=round(_neutral, 4))
except Exception as _e:
    traceback.print_exc()
    _bl['error'] = '%s: %s' % (type(_e).__name__, str(_e)[:500])
    if not STRESS and Path('submission_ours.csv').exists():
        shutil.copy('submission_ours.csv', 'submission.csv')
        _bl['mode'] = 'ours only (blend failed)'
Path('blend_receipt.json').write_text(json.dumps(_bl, indent=1, default=str))
print('BLEND', json.dumps(_bl, default=str))
pd.read_csv('submission.csv').head()